# Question 3: Indexing with MinSearch

Recreate Question 2's chunks from `books_text/`, convert their lists of lines into strings, and index all books with MinSearch. This notebook runs independently of Question 2.

Run the cells in order using the project's Python environment. `minsearch` and `gitsource` are already project dependencies. If needed, install them from the repository root with `uv add minsearch gitsource`.

In [ ]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

# Support running from the repository root or homework/.
project_root = Path.cwd()
if not (project_root / "books_text").is_dir():
    project_root = project_root.parent

books_text_dir = project_root / "books_text"
markdown_files = sorted(books_text_dir.glob("*.md"))
if not markdown_files:
    raise FileNotFoundError(f"No Markdown books found in {books_text_dir}")

## Load and chunk all books

Keep non-empty lines and apply the same 100-line window and 50-line step as Question 2.

In [ ]:
books = []
for markdown_path in markdown_files:
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    books.append({
        "source": markdown_path.name,
        "content": [line for line in lines if line.strip()],
    })

chunks = chunk_documents(books, size=100, step=50)
print(f"Created {len(chunks):,} chunks from {len(books)} books.")

## Prepare searchable documents

MinSearch expects text strings. Join each chunk’s lines with newlines, preserving its source and starting position without modifying the original chunk.

In [ ]:
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)
    return documents


documents = prepare_documents(chunks)

## Build the index and report the answer

In [ ]:
index = Index(text_fields=["content"])
index.fit(documents)

print(f"Number of documents (chunks) indexed: {len(documents)}")